In [26]:
import pandas as pd

voter = "Data/nsn_se15_2023.csv"

df = pd.read_csv(voter)
print(df.shape)   # (864425, 11)
print(df.dtypes)
df.head(5)

(864425, 11)
uid             str
birth_year    int64
sex             str
ethnicity       str
state           str
parlimen        str
dun             str
dm_vr           str
dm              str
pm              str
saluran       int64
dtype: object


,uid,birth_year,sex,ethnicity,state,parlimen,dun,dm_vr,dm,pm,saluran
0,VOKNZLPHGP,1963,Male,Malay,Negeri Sembilan,P.126 JELEBU,N01 CHENNAH,126/01/05 Pekan Titi,126/01/00 Undi Awal,Khemah A IPD Jelebu,1
1,WHCVNPJUCD,1965,Male,Malay,Negeri Sembilan,P.126 JELEBU,N01 CHENNAH,126/01/05 Pekan Titi,126/01/00 Undi Awal,Khemah A IPD Jelebu,1
2,YYBWEDQIUK,1966,Male,Malay,Negeri Sembilan,P.126 JELEBU,N01 CHENNAH,126/01/03 Simpang Durian,126/01/00 Undi Awal,Khemah A IPD Jelebu,1
3,FXWUHPWPPG,1968,Male,Malay,Negeri Sembilan,P.126 JELEBU,N01 CHENNAH,126/01/05 Pekan Titi,126/01/00 Undi Awal,Khemah A IPD Jelebu,1
4,VQKCMMVLKX,1976,Male,Malay,Negeri Sembilan,P.126 JELEBU,N01 CHENNAH,126/01/03 Simpang Durian,126/01/00 Undi Awal,Khemah A IPD Jelebu,1


In [27]:
results = "Data/ns_prn2023_results.csv"

df2 = pd.read_csv(results)
print(df2.shape)   # (864425, 11)
# print(df2.dtypes)
df2.head(5)

(1827, 15)


,PARLIMEN,DUN,NO. KOD DAERAH MENGUNDI,NAMA PUSAT MENGUNDI,SALURAN,KERTAS UNDI DALAM PETI UNDI (A),PN,PH,BN,MUDA,IND,IND.1,JUMLAH UNDI,UNDI YANG DITOLAK (C),KERTAS UNDI TIDAK DIMASUKKAN KE DALAM PETI UNDI (D)
0,P.126 JELEBU,N01 CHENNAH,UNDI POS,NaN,NaN,115,64,29,0,0,0,0,93,8,14
1,P.126 JELEBU,N01 CHENNAH,UNDI AWAL 126 / 01 / 00,KHEMAH A IPD JELEBU,1.0,29,20,9,0,0,0,0,29,0,0
2,P.126 JELEBU,N01 CHENNAH,KAMPONG SUNGAI BULOH 126 / 01 / 01,SEKOLAH KEBANGSAAN SUNGAI BULOH,1.0,325,165,160,0,0,0,0,325,0,0
3,P.126 JELEBU,N01 CHENNAH,KAMPONG SUNGAI BULOH 126 / 01 / 01,SEKOLAH KEBANGSAAN SUNGAI BULOH,2.0,350,229,118,0,0,0,0,347,3,0
4,P.126 JELEBU,N01 CHENNAH,DURIAN TIPUS 126 / 01 / 02,SEKOLAH KEBANGSAAN DURIAN TIPUS,1.0,267,73,192,0,0,0,0,265,2,0


## Aggregate the PRN2023 roll into ethnic composition per polling stream (saluran)

Drop postal/early voters, extract a clean station code, bucket ethnicity into
Malay/Chinese/Indian/Other, and compute each stream's ethnic composition + registered
voter count.

In [28]:
roll = df.copy()

# Drop postal and early voters
is_postal_or_early = roll['dm'].str.contains('Undi Pos|Undi Awal', na=False)
roll_ordinary = roll[~is_postal_or_early].copy()

print(f"dropped {is_postal_or_early.sum()} postal/early voters "
      f"({is_postal_or_early.sum() / len(roll) * 100:.2f}% of {len(roll)})")

# Extract a clean station code from dm, e.g. "126/01/01 Kampong Sungai Buloh" -> "126/01/01"
roll_ordinary['dm_code'] = (
    roll_ordinary['dm']
    .str.extract(r'^(\d+\s*/\s*\d+\s*/\s*\d+)')[0]
    .str.replace(r'\s+', '', regex=True)
)
assert roll_ordinary['dm_code'].isna().sum() == 0, "some ordinary rows failed dm_code extraction"

# Bucket ethnicity into the paper's four categories (Orang Asli / Bumi Sabah / Bumi / Sarawak fold into "Other")
ethnicity_map = {
    'Malay': 'malay',
    'Chinese': 'chinese',
    'Indian': 'indian',
    'Other': 'other',
    'Orang Asli': 'other',
    'Bumi Sabah': 'other',
    'Bumi Sarawak': 'other',
}
roll_ordinary['No.'] = roll_ordinary['ethnicity'].map(ethnicity_map)
assert roll_ordinary['No.'].isna().sum() == 0, "unmapped ethnicity value found"

# Group by (dun, dm_code, saluran) and count voters per ethnic group
stream_counts = (
    roll_ordinary
    .groupby(['dun', 'dm_code', 'saluran', 'No.'])
    .size()
    .unstack('No.', fill_value=0)
)

# Convert to percentages + keep registered count
stream_ethnic = stream_counts.copy()
stream_ethnic['n_registered'] = stream_counts.sum(axis=1)
for col in ['malay', 'chinese', 'indian', 'other']:
    stream_ethnic[f'pct_{col}'] = stream_counts[col] / stream_ethnic['n_registered']

stream_ethnic = stream_ethnic.reset_index()

# Check percentages sum to ~1.0
pct_cols = [f'pct_{c}' for c in ['malay', 'chinese', 'indian', 'other']]
pct_sum = stream_ethnic[pct_cols].sum(axis=1)
bad = stream_ethnic[(pct_sum - 1).abs() > 0.02]
assert len(bad) == 0, f"{len(bad)} streams have ethnic percentages not summing to ~1.0"

print(stream_ethnic.shape)
stream_ethnic.head(5)

dropped 26225 postal/early voters (3.03% of 864425)
(1405, 12)


No.,dun,dm_code,saluran,chinese,indian,malay,other,n_registered,pct_malay,pct_chinese,pct_indian,pct_other
0,N01 CHENNAH,126/01/01,1,1,0,447,2,450,0.993333,0.002222,0.000000,0.004444
1,N01 CHENNAH,126/01/01,2,0,0,535,4,539,0.992579,0.000000,0.000000,0.007421
2,N01 CHENNAH,126/01/02,1,248,17,149,36,450,0.331111,0.551111,0.037778,0.080000
3,N01 CHENNAH,126/01/02,2,55,8,124,156,343,0.361516,0.160350,0.023324,0.454810
4,N01 CHENNAH,126/01/03,1,123,22,302,3,450,0.671111,0.273333,0.048889,0.006667


## A4: Melt PRN2023 results into long format and merge with ethnic composition

Turn the wide `PN`/`PH`/`BN`/`MUDA`/`IND`/`IND.1` columns into one row per
party-per-stream, drop postal/early rows (same exclusion as A3), compute each row's
`vote_share`, then join against `stream_ethnic` on `(dun, dm_code, saluran)`.

In [29]:
results = df2.copy()
results['DUN'] = results['DUN'].str.strip()

# Drop postal and early rows (same exclusion as A3, see CLAUDE.md for why)
is_postal = results['NO. KOD DAERAH MENGUNDI'] == 'UNDI POS'
is_early = results['NO. KOD DAERAH MENGUNDI'].str.contains('UNDI AWAL', na=False)
results_ordinary = results[~(is_postal | is_early)].copy()

print(f"dropped {(is_postal | is_early).sum()} postal/early rows out of {len(results)}")

dropped 116 postal/early rows out of 1827


In [30]:
# Extract a clean station code from the messy "NAME 126 / 01 / 01" text
# (\s already matches the embedded newlines noted in CLAUDE.md)
code_parts = results_ordinary['NO. KOD DAERAH MENGUNDI'].str.extract(r'(\d+)\s*/\s*(\d+)\s*/\s*(\d+)')
results_ordinary['dm_code'] = code_parts[0] + '/' + code_parts[1] + '/' + code_parts[2]
assert results_ordinary['dm_code'].isna().sum() == 0, "some ordinary rows failed dm_code extraction"

results_ordinary['saluran'] = results_ordinary['SALURAN'].astype(int)

In [31]:
# Some stations report multiple "meja" (ballot box) rows for the same saluran when a
# stream is large enough to need more than one box - e.g. Kampung Serting Ilir's
# saluran 1 is split across two rows. Sum these up to one row per physical stream so
# the granularity matches stream_ethnic (one row per (dun, dm_code, saluran)).
num_cols = ['PN', 'PH', 'BN', 'MUDA', 'IND', 'IND.1', 'JUMLAH UNDI',
            'KERTAS UNDI DALAM PETI UNDI (A)', 'UNDI YANG DITOLAK (C)']
results_collapsed = (
    results_ordinary
    .groupby(['DUN', 'dm_code', 'saluran'], as_index=False)[num_cols]
    .sum()
)

print(f"collapsed {len(results_ordinary)} rows -> {len(results_collapsed)} streams "
      f"(expect 1405, matching stream_ethnic)")
assert len(results_collapsed) == len(stream_ethnic), "stream count mismatch after collapsing meja rows"

collapsed 1711 rows -> 1405 streams (expect 1405, matching stream_ethnic)


In [32]:
# Melt party columns into long form. A "0" already means "didn't contest here" (fixed
# columns are populated with 0, not left blank/NaN), so filter votes > 0 after melting.
# NOTE: IND and IND.1 are two DISTINCT independents where a seat has both (N10 Nilai,
# N13 Sikamat) - kept as separate "party" labels here, not summed together.
party_cols = ['PN', 'PH', 'BN', 'MUDA', 'IND', 'IND.1']
results_long = results_collapsed.melt(
    id_vars=['DUN', 'dm_code', 'saluran', 'JUMLAH UNDI'],
    value_vars=party_cols,
    var_name='party',
    value_name='votes',
)
results_long = results_long[results_long['votes'] > 0].copy()
results_long = results_long.rename(columns={'DUN': 'dun', 'JUMLAH UNDI': 'valid_votes'})
results_long['vote_share'] = results_long['votes'] / results_long['valid_votes']

print(results_long.shape)
results_long.head(10)

(3227, 7)


,dun,dm_code,saluran,valid_votes,party,votes,vote_share
0,N01 CHENNAH,126/01/01,1,325,PN,165,0.507692
1,N01 CHENNAH,126/01/01,2,347,PN,229,0.659942
2,N01 CHENNAH,126/01/02,1,265,PN,73,0.275472
3,N01 CHENNAH,126/01/02,2,215,PN,76,0.353488
4,N01 CHENNAH,126/01/03,1,282,PN,98,0.347518
5,N01 CHENNAH,126/01/03,2,344,PN,131,0.380814
6,N01 CHENNAH,126/01/03,3,329,PN,162,0.492401
7,N01 CHENNAH,126/01/03,4,290,PN,142,0.489655
8,N01 CHENNAH,126/01/04,1,309,PN,112,0.362460
9,N01 CHENNAH,126/01/04,2,404,PN,209,0.517327


In [33]:
# Merge with ethnic composition per stream; use an outer join + indicator so mismatches
# would be visible rather than silently dropped.
merged = results_long.merge(
    stream_ethnic,
    on=['dun', 'dm_code', 'saluran'],
    how='outer',
    indicator=True,
)

print(merged['_merge'].value_counts())
unmatched = merged[merged['_merge'] != 'both']
assert len(unmatched) == 0, f"{len(unmatched)} rows failed to match between results and ethnic composition"
merged = merged.drop(columns='_merge')

print(merged.shape)
merged.head(10)

_merge
both          3227
left_only        0
right_only       0
Name: count, dtype: int64
(3227, 16)


,dun,dm_code,saluran,valid_votes,party,votes,vote_share,chinese,indian,malay,other,n_registered,pct_malay,pct_chinese,pct_indian,pct_other
0,N01 CHENNAH,126/01/01,1,325,PN,165,0.507692,1,0,447,2,450,0.993333,0.002222,0.000000,0.004444
1,N01 CHENNAH,126/01/01,1,325,PH,160,0.492308,1,0,447,2,450,0.993333,0.002222,0.000000,0.004444
2,N01 CHENNAH,126/01/01,2,347,PN,229,0.659942,0,0,535,4,539,0.992579,0.000000,0.000000,0.007421
3,N01 CHENNAH,126/01/01,2,347,PH,118,0.340058,0,0,535,4,539,0.992579,0.000000,0.000000,0.007421
4,N01 CHENNAH,126/01/02,1,265,PN,73,0.275472,248,17,149,36,450,0.331111,0.551111,0.037778,0.080000
5,N01 CHENNAH,126/01/02,1,265,PH,192,0.724528,248,17,149,36,450,0.331111,0.551111,0.037778,0.080000
6,N01 CHENNAH,126/01/02,2,215,PN,76,0.353488,55,8,124,156,343,0.361516,0.160350,0.023324,0.454810
7,N01 CHENNAH,126/01/02,2,215,PH,139,0.646512,55,8,124,156,343,0.361516,0.160350,0.023324,0.454810
8,N01 CHENNAH,126/01/03,1,282,PN,98,0.347518,123,22,302,3,450,0.671111,0.273333,0.048889,0.006667
9,N01 CHENNAH,126/01/03,1,282,PH,184,0.652482,123,22,302,3,450,0.671111,0.273333,0.048889,0.006667
